In [0]:
# --------------------------------------------------------------
# Set environment variables for the pipeline run
# --------------------------------------------------------------
# These values are retrieved from the master pipeline task 
# (superstore_pipeline_master_run_id_init) using dbutils.jobs.taskValues.
# They ensure that the ETL, Mart, Features, KPI, and Dashboard code
# runs in the correct environment (dev/qa/prod) without hardcoding paths.
# SUPERSTORE_ENV, SUPERSTORE_PIPELINE_NAME, and SUPERSTORE_PIPELINE_VERSION
# are set for downstream modules to dynamically resolve catalog/schema.
# --------------------------------------------------------------
import os

env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -------------------------------
# Environment Config
# -------------------------------
import sys
sys.path.append("/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/superstore_shared_utilities")
from superstore_platform_config import get_catalog, get_gold_schema, get_mart_schema

# Dynamically get environment-aware catalog and schemas
catalog = get_catalog()            # e.g., superstore_catalog
gold_schema = get_gold_schema()    # e.g., dev_gold
mart_schema = get_mart_schema() 

from superstore_logger import get_superstore_logger, log_event
# -----------------------------
# log_event Setup 
# -----------------------------
# Initialize log_event to capture events 
logger_mart_sales_daily = get_superstore_logger("superstore_mart_sales_daily")

# -------------------------------
# Function to check if the table exists
# -------------------------------
def check_table_exists(catalog, schema, table_name):
    try:
        full_table_name = f"{catalog}.{schema}.{table_name}"
        return spark.catalog.tableExists(full_table_name)
    except Exception as e:
        log_event(
            logger_mart_sales_daily,
            "ERROR",
            f"Error checking table existence: {e}",
            layer="Feature Engineering"
        )
        return False

# -------------------------------
# Required table to create or update features Table (Environment-aware)
# -------------------------------
# Check if the required tables exist before creating the feature table
required_tables = [
    f"{catalog}.{gold_schema}.dim_customers",
    f"{catalog}.{gold_schema}.facts_orders",
    f"{catalog}.{gold_schema}.facts_sales"
]

missing_tables = []

# Check each table
for table in required_tables:
    if not check_table_exists(catalog, gold_schema, table.split('.')[-1]):
        missing_tables.append(table)

if missing_tables:
    log_event(logger_mart_sales_daily, "ERROR", f"Missing tables: {', '.join(missing_tables)}. Cannot create mart sales daily table.", layer="Mart")
else:
  # -------------------------------
  # Create mart Table (Environment-aware)
  # -------------------------------
  spark.sql(f"""
  CREATE OR REPLACE TABLE {catalog}.{mart_schema}.mart_sales_daily AS
  SELECT
    o.order_date,
    o.customer_id,
    s.product_id,
    c.region,
    c.segment,
    p.category,
    p.sub_category,
    o.ship_mode,

    SUM(s.sales) AS total_sales,
    SUM(s.profit) AS total_profit,
    SUM(s.quantity) AS total_quantity,
    AVG(s.discount) AS avg_discount,
    COUNT(DISTINCT o.order_id) AS order_count,

    CASE 
    WHEN SUM(s.sales) = 0 THEN NULL 
    ELSE ROUND(SUM(s.profit) / SUM(s.sales), 4) 
    END AS profit_margin

  FROM {catalog}.{gold_schema}.facts_orders o
  JOIN {catalog}.{gold_schema}.facts_sales s 
  ON o.order_id = s.order_id

  -- FIXED: use o.customer_id
  JOIN {catalog}.{gold_schema}.dim_customers c 
      ON o.customer_id = c.customer_id 
      AND c.is_current = true

  JOIN {catalog}.{gold_schema}.dim_products p 
    ON s.product_id = p.product_id 
    AND p.is_current = true

  GROUP BY
    o.order_date,
    o.customer_id,
    s.product_id,
    c.region,
    c.segment,
    p.category,
    p.sub_category,
    o.ship_mode
  ;
  """)

log_event(logger_mart_sales_daily, "INFO", "Mart mart sales daily table created successfully.", layer="Mart")
